# 29 -- More training data from the same site: buffer only at the validation boundary

The spatial-block split of `09`/`19`/`26` discards every patch within 150 m
of *any* block boundary -- 887 of 1676 patches (53%) -- including boundaries
between two training blocks, where leakage cannot occur. Leakage only
crosses a train--validation boundary, and the standard remedy is to buffer
around the validation fold alone (Roberts et al. 2017).

This notebook keeps the **validation set identical** (the same 255 patches,
asserted) and rebuilds the training set as *every other paired patch whose
edge-to-edge distance from every validation patch is at least
`MIN_SEP_M` = 128 m* -- the minimum separation the original split achieves,
so the leakage margin is unchanged. The R-tree overlap check is repeated and
must return zero.

Configuration: the leading one (`repeat` channels, real attributes, k = 3,
standardised input from this training split's own statistics, PLMS
evaluation), seed 42 and seed 43. Compare against the 534-patch runs:
0.3700 (seed 42) / 0.3558 (seed 43), seed s.d. 0.043.

| tag | training patches | seed |
|---|---|---|
| `std_realattrs_valbuf` | all eligible (expected ~1000+) | 42 |
| `std_realattrs_valbuf_seed43` | same | 43 |

Resumable as in `26`. ~1.5 h training + ~40 min evaluation per run.


In [1]:
import os, sys, json, random, time, datetime as dt
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import rasterio

assert torch.cuda.is_available(), 'CUDA is required.'
DEVICE = torch.device('cuda')
torch.backends.cudnn.benchmark = True
print('GPU:', torch.cuda.get_device_name(0))

GPU: NVIDIA GeForce RTX 4090


In [2]:
WORKING_REPO = Path('/cs/student/project_msc/2025/aibh/jiayiche')
TESSA_REPO = WORKING_REPO / 'tessa_baseline'
CHECKPOINT_DIR = WORKING_REPO / 'checkpoints'
OUTPUT_DIR = WORKING_REPO / 's1_training_outputs'
LIDAR_DIR = WORKING_REPO / 'input_data' / 'lidar_patches_tuk_tessa'
S1_DIRS = {'IW': WORKING_REPO / 'input_data' / 's1_patches_tuk_pcrtc',
           'EW': WORKING_REPO / 'input_data' / 's1_patches_tuk_ew'}
LIDAR_SURVEY_DATE = dt.date(2024, 4, 16)

TARGET_HW = (256, 256); BATCH_SIZE = 8; EPOCHS = 100; TIMESTEPS = 1000; LEARNING_RATE = 1e-4
VAL_FRACTION = 0.15; SPLIT_SEED = 42; NOISE_SCHEDULE = 'linear'; ATTENTION_VARIANT = 'default'
BLOCK_SIZE_M, BUFFER_M = 1024.0, 150.0
NUM_WORKERS = 4
EVAL_SAMPLER_NAME = 'plms'
MIN_SEP_M = 128.0   # minimum edge-to-edge separation train<->val, equal to what the block split achieves

CONFIGS = [
    dict(tag='std_realattrs_valbuf',        channels='repeat', attrs='real', k=3, data='IW', seed=42),
    dict(tag='std_realattrs_valbuf_seed43', channels='repeat', attrs='real', k=3, data='IW', seed=43),
]
def ckpt_path(tag): return CHECKPOINT_DIR / f's1_tuk_pcrtc_{tag}_unet_best.pth'
def metrics_path(tag): return OUTPUT_DIR / f's1_pcrtc_{tag}_{EVAL_SAMPLER_NAME}_validation_metrics.json'
def history_path(tag): return OUTPUT_DIR / f's1_pcrtc_{tag}_history.json'
def stats_path(tag): return OUTPUT_DIR / f's1_pcrtc_{tag}_norm_stats.json'

for d in S1_DIRS.values(): assert d.exists(), f'missing {d}'

In [3]:
sys.path.insert(0, str(TESSA_REPO))
from src.model.unet import ConditionalUNet
from src.diffusion.scheduler import LinearDiffusionScheduler, CosineDiffusionScheduler
from src.diffusion.sampling import p_sample_loop_ddim, p_sample_loop_plms
from src.utils.recon_metrics import rmse, bias, sigma_error, normal_angle_error, average_jsd_multiscale, log_psd_rmse, zncc
import inspect
assert 'cond_channels_per_view' in inspect.signature(ConditionalUNet.__init__).parameters, \
    'tessa_baseline ConditionalUNet lacks cond_channels_per_view -- apply the workstation patch from 15/16 first'

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

EVAL_SAMPLER = {'plms': p_sample_loop_plms, 'ddim': p_sample_loop_ddim}[EVAL_SAMPLER_NAME]
print('evaluation sampler:', EVAL_SAMPLER_NAME)

evaluation sampler: plms


## Data (as `26`), plus the validation-buffer split

In [4]:
from scipy.ndimage import median_filter

def load_sar_db(time_path, despeckle=0):
    with rasterio.open(time_path) as src:
        sar = src.read()[:2].astype(np.float32)
    sar = np.nan_to_num(sar, nan=0.0, posinf=0.0, neginf=0.0)
    if despeckle:
        sar = median_filter(sar, size=(1, despeckle, despeckle))   # per band, on linear power, at native resolution, before dB
    return 10.0 * np.log10(np.maximum(sar, 1e-12))

def build_real_attrs(s1_path, times, context_k):
    attrs_list = json.load(open(s1_path / 'attrs.json')) if (s1_path / 'attrs.json').exists() else []
    vecs = []
    for t in times:
        idx = int(t.stem[1:]); a = attrs_list[idx] if idx < len(attrs_list) else {}
        age = (dt.date.fromisoformat(a['acquisition_date']) - LIDAR_SURVEY_DATE).days / 30.0 if a.get('acquisition_date') else 0.0
        vecs.append([age, 1.0 if a.get('orbit_direction') == 'ASCENDING' else 0.0, (a.get('relative_orbit_number') or 0) / 175.0, 0, 0, 0, 0, 0])
    return torch.tensor(vecs, dtype=torch.float32).flatten()

class LidarS1Dataset(Dataset):
    def __init__(self, s1_dir, lidar_dir, patch_ids, context_k, channels, attrs_mode, norm_stats, despeckle=0):
        self.s1_dir, self.lidar_dir, self.patch_ids = Path(s1_dir), Path(lidar_dir), list(patch_ids)
        self.context_k, self.channels, self.attrs_mode, self.norm_stats, self.despeckle = context_k, channels, attrs_mode, norm_stats, despeckle
    def __len__(self): return len(self.patch_ids)
    def __getitem__(self, i):
        pid = self.patch_ids[i]
        with rasterio.open(self.lidar_dir / f'lidar_patch_{pid}.tif') as src:
            raw = src.read().astype(np.float32)
        target = raw[0]; mask = (raw[1] > 0.5) if raw.shape[0] > 1 else np.isfinite(target)
        target = np.nan_to_num(target, nan=0.0, posinf=0.0, neginf=0.0)
        pm = float(target[mask].sum() / max(1, int(mask.sum()))); target = (target - pm) * mask
        s1_path = self.s1_dir / f's1_patch_{pid}'
        times = sorted(s1_path.glob('t*.tif'))[:self.context_k]
        if len(times) < self.context_k: raise RuntimeError(f'{s1_path} has fewer than {self.context_k} views')
        views = []
        mean, std = self.norm_stats
        for t in times:
            sar = (load_sar_db(t, self.despeckle) - mean[:, None, None]) / std[:, None, None]
            st = F.interpolate(torch.from_numpy(sar).unsqueeze(0), size=TARGET_HW, mode='bilinear', align_corners=False).squeeze(0)
            views.append(st.repeat(2, 1, 1) if self.channels == 'repeat' else st)
        cond = torch.cat(views, dim=0)
        attrs = build_real_attrs(s1_path, times, self.context_k) if self.attrs_mode == 'real' else torch.zeros(8 * self.context_k)
        return {'lidar': torch.from_numpy(target).unsqueeze(0).float(), 'mask': torch.from_numpy(mask), 's1': cond.float(),
                'attrs': attrs, 'patch_mean': torch.tensor(pm), 'patch_id': pid}

def spatial_split(s1_dir):
    lidar_ids = {p.stem.split('_')[-1] for p in LIDAR_DIR.glob('lidar_patch_*.tif')}
    s1_ids = {p.name.split('_')[-1] for p in Path(s1_dir).glob('s1_patch_*') if p.is_dir()}
    paired = sorted(lidar_ids & s1_ids)
    blocks, dropped = {}, []
    for pid in paired:
        with rasterio.open(LIDAR_DIR / f'lidar_patch_{pid}.tif') as src: b = src.bounds
        cx, cy = (b.left + b.right) / 2, (b.bottom + b.top) / 2
        bx, by = int(cx // BLOCK_SIZE_M), int(cy // BLOCK_SIZE_M)
        d = min(cx - bx * BLOCK_SIZE_M, (bx + 1) * BLOCK_SIZE_M - cx, cy - by * BLOCK_SIZE_M, (by + 1) * BLOCK_SIZE_M - cy)
        (dropped if d < BUFFER_M else blocks.setdefault((bx, by), [])).append(pid)
    ids = list(blocks); random.Random(SPLIT_SEED).shuffle(ids)
    target_val = int(len(paired) * VAL_FRACTION); val, train, n = [], [], 0
    for bid in ids:
        if n < target_val: val.extend(blocks[bid]); n += len(blocks[bid])
        else: train.extend(blocks[bid])
    assert not (set(train) & set(val))
    return train, val, len(dropped)

def compute_stats(s1_dir, train_ids, k, despeckle=0):
    sums = np.zeros(2); sqs = np.zeros(2); count = 0
    for pid in train_ids:
        for t in sorted((Path(s1_dir) / f's1_patch_{pid}').glob('t*.tif'))[:k]:
            sar = load_sar_db(t, despeckle); sums += sar.reshape(2, -1).sum(1); sqs += (sar.reshape(2, -1) ** 2).sum(1); count += sar.shape[1] * sar.shape[2]
    mean = sums / count; std = np.sqrt(np.maximum(sqs / count - mean ** 2, 1e-12))
    return mean.astype(np.float32), std.astype(np.float32)

def bounds_of(pid):
    with rasterio.open(LIDAR_DIR / f'lidar_patch_{pid}.tif') as src: b = src.bounds
    return (b.left, b.bottom, b.right, b.top)

def edge_distance(a, b):
    dx = max(b[0] - a[2], a[0] - b[2], 0.0); dy = max(b[1] - a[3], a[1] - b[3], 0.0)
    return (dx * dx + dy * dy) ** 0.5

def valbuffer_split(s1_dir):
    """Same validation set as spatial_split; training = every other paired patch >= MIN_SEP_M from all validation patches."""
    train_blk, val, dropped_blk = spatial_split(s1_dir)
    lidar_ids = {p.stem.split('_')[-1] for p in LIDAR_DIR.glob('lidar_patch_*.tif')}
    s1_ids = {p.name.split('_')[-1] for p in Path(s1_dir).glob('s1_patch_*') if p.is_dir()}
    paired = sorted(lidar_ids & s1_ids)
    global BOUNDS
    BOUNDS = {pid: bounds_of(pid) for pid in paired}   # read each file once
    vb = {pid: BOUNDS[pid] for pid in val}
    train, dropped = [], []
    for pid in paired:
        if pid in vb: continue
        b = BOUNDS[pid]
        (train if min(edge_distance(b, v) for v in vb.values()) >= MIN_SEP_M else dropped).append(pid)
    assert set(train_blk) <= set(train), 'block-split training patches must all remain eligible'
    return train, val, len(dropped)

SPLITS = {'IW': valbuffer_split(S1_DIRS['IW'])}
tr, va, dr = SPLITS['IW']
print(f'validation-buffer split: train={len(tr)} val={len(va)} dropped={dr}  (block split: train=534 val=255 dropped=887)')
assert len(va) == 255, 'validation set must be identical to 09/19/26'
assert set(va) == set(spatial_split(S1_DIRS['IW'])[1])

# leakage check: zero overlaps and minimum separation, exactly as in 27
from shapely.geometry import box
from shapely.strtree import STRtree
vboxes = [box(*BOUNDS[p]) for p in va]; tree = STRtree(vboxes)
overlaps = sum(1 for p in tr if any(vboxes[k].intersects(box(*BOUNDS[p])) and not vboxes[k].touches(box(*BOUNDS[p])) for k in tree.query(box(*BOUNDS[p]))))
min_sep = min(min(edge_distance(BOUNDS[p], BOUNDS[v]) for v in va) for p in tr)
print(f'{overlaps} / {len(tr)} training patches overlap a validation patch (must be 0); min train-val separation {min_sep:.1f} m')
assert overlaps == 0 and min_sep >= MIN_SEP_M
json.dump({'train': tr, 'val': va, 'n_dropped': dr, 'min_sep_m': min_sep, 'rule': f'val identical to block split; train = all others >= {MIN_SEP_M} m from every val patch'},
          open(OUTPUT_DIR / 's1_pcrtc_valbuffer_split.json', 'w'), indent=1)


validation-buffer split: train=1075 val=255 dropped=346  (block split: train=534 val=255 dropped=887)
0 / 1075 training patches overlap a validation patch (must be 0); min train-val separation 128.0 m


## Train + evaluate one configuration (as `26`; statistics always from this training split)

In [5]:
def masked_mse(pred, target, mask):
    valid = mask.bool().unsqueeze(1); return ((pred - target) ** 2)[valid].mean()

def run_config(cfg):
    tag, k = cfg['tag'], cfg['k']
    epochs = cfg.get('epochs', EPOCHS)
    if metrics_path(tag).exists():
        print(f'[{tag}] metrics exist -- skipping'); return json.load(open(metrics_path(tag)))
    s1_dir = S1_DIRS[cfg['data']]; train_ids, val_ids, _ = SPLITS[cfg['data']]
    despeckle = cfg.get('despeckle', 0)
    # the training split differs from 19's, so the standardisation statistics are recomputed from it (training ids only)
    stats = compute_stats(s1_dir, train_ids, k, despeckle)
    json.dump({'mean': stats[0].tolist(), 'std': stats[1].tolist(), 'n_train': len(train_ids), 'k': k, 'despeckle': despeckle}, open(stats_path(tag), 'w'), indent=2)
    print(f'[{tag}] train={len(train_ids)} val={len(val_ids)}  stats VV {stats[0][0]:.2f}/{stats[1][0]:.2f}  VH {stats[0][1]:.2f}/{stats[1][1]:.2f}')

    seed_everything(cfg['seed'])
    ds_tr = LidarS1Dataset(s1_dir, LIDAR_DIR, train_ids, k, cfg['channels'], cfg['attrs'], stats, despeckle)
    ds_va = LidarS1Dataset(s1_dir, LIDAR_DIR, val_ids,   k, cfg['channels'], cfg['attrs'], stats, despeckle)
    train_loader = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(ds_va, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    cpv = 4 if cfg['channels'] == 'repeat' else 2
    model = ConditionalUNet(in_channels=1, cond_channels=cpv * k, attr_dim=8 * k, base_channels=128, embed_dim=256, unet_depth=4,
                            attention_variant=ATTENTION_VARIANT, cond_k=k, cond_channels_per_view=cpv).to(DEVICE)
    scheduler = LinearDiffusionScheduler(TIMESTEPS, device=DEVICE) if NOISE_SCHEDULE == 'linear' else CosineDiffusionScheduler(TIMESTEPS, device=DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE); scaler = GradScaler()

    history = {'train_loss': [], 'val_loss': []}; best = float('inf'); t0 = time.time()
    for epoch in range(epochs):
        model.train(); tr_tot = 0.0
        for b in train_loader:
            target, cond, attrs, mask = (b[key].to(DEVICE, non_blocking=True) for key in ('lidar', 's1', 'attrs', 'mask'))
            ts = torch.randint(0, TIMESTEPS, (target.size(0),), device=DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with autocast():
                loss = masked_mse(model(scheduler.q_sample(target, ts), cond, attrs, ts), target, mask)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update(); tr_tot += loss.item()
        model.eval(); va_tot = 0.0
        with torch.no_grad():
            for b in val_loader:
                target, cond, attrs, mask = (b[key].to(DEVICE, non_blocking=True) for key in ('lidar', 's1', 'attrs', 'mask'))
                ts = torch.randint(0, TIMESTEPS, (target.size(0),), device=DEVICE)
                with autocast():
                    va_tot += masked_mse(model(scheduler.q_sample(target, ts), cond, attrs, ts), target, mask).item()
        tr, va = tr_tot / max(1, len(train_loader)), va_tot / max(1, len(val_loader))
        history['train_loss'].append(tr); history['val_loss'].append(va)
        if (epoch + 1) % 10 == 0 or epoch == 0: print(f'[{tag}] epoch {epoch+1:03d}/{epochs} train={tr:.6f} val={va:.6f}  {(time.time()-t0)/60:.0f} min')
        if va < best:
            best = va
            torch.save({'model_state_dict': model.state_dict(), 'config': {**cfg, 'timesteps': TIMESTEPS, 'noise_schedule': NOISE_SCHEDULE, 'standardised': True},
                        'epoch': epoch + 1, 'val_loss': va}, ckpt_path(tag))
    json.dump(history, open(history_path(tag), 'w'))

    # evaluate best checkpoint with EVAL_SAMPLER (PLMS, the reference study's choice; 24 showed +0.05..0.07 over DDIM)
    model.load_state_dict(torch.load(ckpt_path(tag), map_location=DEVICE)['model_state_dict']); model.eval()
    seed_everything(SPLIT_SEED); rows = []
    with torch.no_grad():
        for b in val_loader:
            target, cond, attrs = b['lidar'].to(DEVICE), b['s1'].to(DEVICE), b['attrs'].to(DEVICE)
            mask = b['mask'].to(DEVICE).bool(); pm = b['patch_mean'].to(DEVICE).view(-1, 1, 1, 1)
            pred = EVAL_SAMPLER(model, scheduler, target.shape, cond, attrs, DEVICE)
            gt_abs, pred_abs = target + pm, pred + pm
            for i, pid in enumerate(b['patch_id']):
                g, p, m = gt_abs[i], pred_abs[i], mask[i]
                gv, pv = g.squeeze()[m].cpu().numpy(), p.squeeze()[m].cpu().numpy()
                rows.append({'patch_id': pid, 'rmse_m': float(rmse(g, p, m)), 'bias_m': float(bias(g, p, m)),
                             'sigma_error_pct': float(sigma_error(g, p, m)),
                             'normal_angle_error_deg': float(normal_angle_error(g, p, m, pixel_size=1.0, degrees=True)),
                             'jsd': float(average_jsd_multiscale(g, p, pixel_size=1.0, mask=m)),
                             'psd_rmse': float(log_psd_rmse(g, p, pixel_size=1.0, mask=m)), 'zncc': float(zncc(g, p, m)),
                             'gt_std_val': float(gv.std()), 'pred_std_val': float(pv.std())})
    json.dump(rows, open(metrics_path(tag), 'w'), indent=2)
    mean = {key: float(np.nanmean([r[key] for r in rows])) for key in rows[0] if key != 'patch_id'}
    print(f'[{tag}] DONE  ZNCC {mean["zncc"]:+.4f}  RMSE {mean["rmse_m"]:.4f}  sig% {mean["sigma_error_pct"]:.1f}  pred/gt {mean["pred_std_val"]:.4f}/{mean["gt_std_val"]:.4f}  ({(time.time()-t0)/3600:.1f} h)')
    del model, optimizer; torch.cuda.empty_cache()
    return rows

## Run the queue (re-run after any interruption; finished configurations are skipped)

In [6]:
results = {}
for cfg in CONFIGS:
    try:
        results[cfg['tag']] = run_config(cfg)
    except Exception as exc:
        print(f'[{cfg["tag"]}] FAILED: {type(exc).__name__}: {exc}')
        raise

[std_realattrs_valbuf] train=1075 val=255  stats VV -18.19/2.59  VH -24.02/2.61


/tmp/ipykernel_175954/4079872735.py:25: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE); scaler = GradScaler()
/tmp/ipykernel_175954/4079872735.py:34: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_175954/4079872735.py:42: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


[std_realattrs_valbuf] epoch 001/100 train=0.032586 val=0.018693  0 min
[std_realattrs_valbuf] epoch 010/100 train=0.013331 val=0.015518  5 min
[std_realattrs_valbuf] epoch 020/100 train=0.012655 val=0.014611  9 min
[std_realattrs_valbuf] epoch 030/100 train=0.012674 val=0.013969  13 min
[std_realattrs_valbuf] epoch 040/100 train=0.013113 val=0.014119  17 min
[std_realattrs_valbuf] epoch 050/100 train=0.012151 val=0.013649  21 min
[std_realattrs_valbuf] epoch 060/100 train=0.012024 val=0.015149  26 min
[std_realattrs_valbuf] epoch 070/100 train=0.012587 val=0.015476  30 min
[std_realattrs_valbuf] epoch 080/100 train=0.012026 val=0.015602  34 min
[std_realattrs_valbuf] epoch 090/100 train=0.012121 val=0.014871  38 min
[std_realattrs_valbuf] epoch 100/100 train=0.009866 val=0.014318  42 min
[std_realattrs_valbuf] DONE  ZNCC +0.3059  RMSE 0.1879  sig% 20.3  pred/gt 0.1493/0.1721  (1.4 h)
[std_realattrs_valbuf_seed43] train=1075 val=255  stats VV -18.19/2.59  VH -24.02/2.61
[std_realattrs_

## Summary against the 534-patch runs

In [7]:
REF = {  # 534-patch training, same validation set, standardised + PLMS
    'seed 42 (27)': OUTPUT_DIR / 's1_pcrtc_realattrs_spatialsplit_standardised_plms_validation_metrics.json',
    'seed 43 (26)': OUTPUT_DIR / 's1_pcrtc_std_realattrs_seed43_plms_validation_metrics.json',
    'seed 44 (26)': OUTPUT_DIR / 's1_pcrtc_std_realattrs_seed44_plms_validation_metrics.json',
}
NEW = {'valbuf seed 42': metrics_path('std_realattrs_valbuf'), 'valbuf seed 43': metrics_path('std_realattrs_valbuf_seed43')}
def mean_of(path):
    if not Path(path).exists(): return None
    rows = json.load(open(path)); return {k: float(np.nanmean([r[k] for r in rows])) for k in rows[0] if k != 'patch_id'}
n_tr = len(SPLITS['IW'][0])
print(f'{"run":<18}{"train n":>8}{"ZNCC":>8}{"RMSE":>8}{"sig%":>7}{"JSD":>8}{"PSD":>8}{"pred std":>9}')
print('-' * 74)
for label, path in list(REF.items()) + list(NEW.items()):
    m = mean_of(path)
    if m is None: print(f'{label:<18}{"--":>8}'); continue
    n = 534 if label in REF else n_tr
    print(f'{label:<18}{n:>8d}{m["zncc"]:>8.4f}{m["rmse_m"]:>8.4f}{m["sigma_error_pct"]:>7.1f}{m["jsd"]:>8.4f}{m["psd_rmse"]:>8.4f}{m["pred_std_val"]:>9.4f}')
ref = [mean_of(p)['zncc'] for p in REF.values() if mean_of(p)]
new = [mean_of(p)['zncc'] for p in NEW.values() if mean_of(p)]
if ref and new:
    print(f'\n534-patch seeds: mean {np.mean(ref):.4f} sd {np.std(ref, ddof=1):.4f}   |   {n_tr}-patch runs: {new}  mean {np.mean(new):.4f}')
    print(f'difference of means {np.mean(new)-np.mean(ref):+.4f}  (single-run 95% threshold vs the 534 mean: 0.214; permissive 0.100)')


run                train n    ZNCC    RMSE   sig%     JSD     PSD pred std
--------------------------------------------------------------------------
seed 42 (27)           534  0.3700  0.1632   37.8  0.1328  1.9660   0.1056
seed 43 (26)           534  0.3558  0.2115   30.4  0.1002  1.3155   0.2024
seed 44 (26)           534  0.2892  0.1850   23.4  0.0935  1.8419   0.1358
valbuf seed 42        1075  0.3059  0.1879   20.3  0.0786  1.4918   0.1493
valbuf seed 43        1075  0.3080  0.1704   36.9  0.1384  1.7673   0.1042

534-patch seeds: mean 0.3384 sd 0.0432   |   1075-patch runs: [0.3059484983045681, 0.30804162939973906]  mean 0.3070
difference of means -0.0314  (single-run 95% threshold vs the 534 mean: 0.214; permissive 0.100)
